# 01 · CMCV 합의와 난도 분류

세 parser의 text 출력을 pairwise similarity로 비교해 Easy/Medium/Hard를 나눈다. 논문의 실제 model·threshold를 재현하지 않는 toy reproduction이다.

**학습 목표**: parser 간 pairwise agreement를 계산하고 CMCV식 난도 label의 의미와 공통 오류 한계를 설명한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `itertools`만 사용하며 외부 패키지는 없다.

In [ ]:
# combinations는 같은 parser 쌍과 자기 자신을 중복 비교하지 않게 한다.
from itertools import combinations

def levenshtein(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        new = [i]
        for j, cb in enumerate(b, 1):
            new.append(min(new[-1] + 1, row[j] + 1, row[j - 1] + (ca != cb)))
        row = new
    return row[-1]

def similarity(a, b):
    a, b = ' '.join(a.casefold().split()), ' '.join(b.casefold().split())
    return 1 - levenshtein(a, b) / max(1, len(a), len(b))

def cmcv_difficulty(outputs):
    scores = [similarity(a, b) for a, b in combinations(outputs, 2)]
    agreement = sum(scores) / len(scores)
    label = 'Easy' if agreement >= .90 else ('Medium' if agreement >= .65 else 'Hard')
    return label, agreement, scores


In [ ]:
cases = [
    ['Total revenue 42M', 'Total revenue 42M', 'Total revenue 42M'],
    ['x squared plus y', 'x^2 + y', 'x2 plus y'],
    ['Assets 100', '<table><td>100</td></table>', 'liabilities 80'],
]
for outputs in cases:
    label, agreement, scores = cmcv_difficulty(outputs)
    print(label, round(agreement, 3), [round(x, 3) for x in scores])
assert cmcv_difficulty(cases[0])[0] == 'Easy'
assert cmcv_difficulty(cases[-1])[0] == 'Hard'


실제 CMCV는 text/edit, table/TEDS, formula/CDM처럼 task에 맞는 metric을 쓴다. 높은 합의는 세 모델의 공통 오류를 발견하지 못하므로 visual judge와 사람 검수가 필요하다.